# 24. Can a forgotten task be brought back?

This is the question the whole project is for. Unlearning drops task 3 to exactly 10%
and stops the model responding to its input. Is the knowledge gone, or only unreachable?

### Why this is not obvious either way

The weights are not stored. A shared **generator** network produces them from a 32-number
**task code**. Unlearning trains only the generator; the task codes are never touched. So
task 3's code still exists exactly as it was, and the generator still serves seven other
tasks well. Nothing was deleted. One mapping was bent out of shape.

Against that, the model now gives the same answer to every image, and that can make the
learning signal vanish: the error has to travel back through near-zero layers that shrink
it on the way.

### Two stopping points, not one

Our accuracy trace puts task 3 at chance by step 6. The method runs 100 steps. The
direction measurement says the weights turn about 7 degrees by step 20 and about 49 by
step 100, so the later steps move the weights a long way after the task is already gone.

This notebook now runs every arm twice, once from a model unlearned for 6 steps and once
from a model unlearned for 100, each unlearned separately so the two never share state.
That turns a single question into a sharper one: do the 94 extra steps remove anything
more, or do they only make the same removal harder to undo?

### The design, and why it needs four arms

Showing that task 3 comes back is not enough on its own. Task 3 might simply be easy to
learn from a handful of images. The question is whether it comes back **faster than it
would from nothing**, which is what would show that something survived.

Every arm starts from the same unlearned model and gets the same small sample of task 3
images and the same number of training steps.

| arm | task code | BatchNorm statistics | what it isolates |
| --- | --- | --- | --- |
| **attacker** | task 3's original | task 3's original | everything an attacker inherits |
| **code only** | task 3's original | reset | how much the old code alone carries |
| **stats only** | a brand new code | task 3's original | how much the old statistics alone carry |
| **fresh start** | a brand new code | reset | learning task 3 from nothing, in this same generator |

**BatchNorm statistics** are running averages a network keeps about the data it has seen.
They matter here because unlearning never touches them: the repository's own code says so,
and the paper's objective covers generated parameters only. So the statistics still carry
information derived from task 3's images. The stats-only arm measures how much that
matters on its own.

### Why accuracy alone is not enough to read these curves

A model that answers the same class for every image scores exactly 10% on a balanced
ten-class test set. So does a model that guesses. Accuracy cannot tell those two apart,
and that is the whole weakness we are documenting in the method's own metrics. Every
measurement here therefore reports how many distinct classes the model actually predicts
alongside its accuracy. One class used means collapse, not forgetting.

### How to read the result

- **attacker clearly above fresh start** → the information survived unlearning. A privacy
  claim for this method does not hold.
- **attacker close to fresh start** → no usable trace is left, and unlearning did remove
  the task-specific mapping.
- **nothing learns in any arm** → the learning signal has vanished. That is ambiguous, not
  evidence of removal, and the write-up must say so.
- **attacker above code only** → a meaningful part of what survived is in the BatchNorm
  statistics, which unlearning does not even attempt to remove. That is worth reporting on
  its own.
- **recovery at 6 steps but not at 100** → the extra steps buy irreversibility rather than
  removal, which is exactly the claim the step-6 trace suggests.

### Scope

One seed, one forgotten task, one order. The recovery images come from task 3's own
training set, so this is the question "can someone holding a few of the original images
restore the task", not "does the model generalise again from unseen data".

## 1. Load the fixed code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "65e2e7c850fd6f1852a99d9e81780c27f799c926"
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading this fixed code revision.")
if IN_COLAB:
    REPO = Path("/content/cark-paired-study")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
        raise RuntimeError("Checkout has edits. Use a fresh runtime.")
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a CARK checkout.")
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    implementation = ["uncle", "pyproject.toml", "requirements.txt"]
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--", *implementation], text=True).strip()
    # A later documentation-only commit may contain the same fixed implementation.
    differences = subprocess.check_output(["git", "-C", str(REPO), "diff", CODE_VERSION, actual, "--", *implementation], text=True).strip()
    if dirty or differences:
        raise RuntimeError("Use a clean checkout of CODE_VERSION, or open in Colab. Local training edits are preserved.")
for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies before running this notebook.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))
from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle import checkpoint
from uncle.config import Config
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.streams import build_tasks
from uncle.study_training import restore_rng, run_request
# The progress panel only draws bars, so it cannot change a result. It
# therefore sits outside the pinned implementation, in scripts/, and this
# checkout is detached to CODE_VERSION, which predates it. Try the checkout
# first, then main, then fall back to ordinary bars rather than failing.
sys.path.insert(0, str(REPO / 'scripts'))
try:
    from dashboard import Dashboard
except ImportError:
    try:
        tools = Path.cwd() / '_tools'
        tools.mkdir(exist_ok=True)
        subprocess.run(['git', '-C', str(REPO), 'fetch', '--quiet', 'origin', 'main'], check=True)
        (tools / 'dashboard.py').write_bytes(subprocess.check_output(
            ['git', '-C', str(REPO), 'show', 'origin/main:scripts/dashboard.py']))
        sys.path.insert(0, str(tools))
        from dashboard import Dashboard
        print('Progress panel fetched from main.')
    except Exception as problem:
        print('Progress panel unavailable, using plain bars:', problem)

        class Dashboard:
            """Just enough of the panel to keep the notebook running."""

            def __init__(self, *args, **kwargs): pass
            def __enter__(self): return self
            def __exit__(self, *args): return False
            def note(self, text): print(text, flush=True)

            def bar(self, total, description, unit='step'):
                from uncle.telemetry import progress_bar
                return progress_bar(total, description)

import torch
print("Fixed implementation revision:", CODE_VERSION)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)

## 2. Settings

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"

ORDER = "order_01"
SEED = 0
FORGET_TASK = "3"

# Two stopping points, not one.
#
# 6 is where our own two-step accuracy trace first puts task 3 at chance, so it
# is the earliest moment the method could honestly stop. 100 is the paper's own
# setting. Running both in one session is the point of this notebook now: it
# asks whether the 94 extra steps remove anything more, or only make the same
# removal harder to undo.
#
# Every arm below is run once per entry here, each from its own freshly
# unlearned model, so the two settings never share state.
UNLEARN_BUDGETS = [6, 100]

# How many images per class the recovery gets. Ten classes, so 20 means 200
# images in total. These come from task 3's own training set.
BUDGETS = [1, 5, 20]

# The same number of gradient steps for every arm and every budget, so the
# comparison is about what survived rather than how much training happened.
RECOVERY_STEPS = 100
# Measured every 10 rather than every 20. The interesting part of these curves
# is the beginning, where a collapsed model spends its dead time before any
# learning signal appears, and every-20 was too coarse to locate that.
MEASURE_EVERY = 10

# Tasks the model has never seen. Order 01 uses 3, 0, 9, 5, 17, 1, 7 and 14,
# and task 15 is learned afterwards, so these are untouched. Each is learned
# from the same unlearned model with a fresh code and the same budget, which
# answers "cheap compared to what?". Several are used because tasks differ in
# difficulty and one would not be convincing.
UNSEEN_TASKS = ["2", "4", "6"]

# Let the generator train. This default has changed, and the reason matters.
#
# It used to be True, following the authors' own recovery protocol, which
# freezes the generator and trains only a new 32-number task code. We have
# since measured that setting directly, on three seeds: a frozen generator
# with a fresh code sits at chance, 10.35 and 10.00 percent against a chance
# level of 10. So with the generator frozen, the fresh-start arm cannot move
# off the floor, and the attacker-versus-fresh-start comparison is being made
# against a control that is pinned by construction rather than by the result.
#
# False lets the generator train. That is both what an attacker could actually
# do and what makes fresh start a real control. Set it back to True only to
# reproduce the authors' stricter protocol, and when you do, read the
# fresh-start arm there as a floor rather than as a comparison.
FREEZE_GENERATOR = False

# Only meaningful when the generator can move. With it frozen the other tasks
# cannot change at all, so the protection term has nothing to do.
PROTECT_DURING_RECOVERY = True

SAVE_TO_R2 = True
RELEASE_GPU_WHEN_DONE = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
ARM_COUNT = 4
RUNS = len(UNLEARN_BUDGETS) * len(BUDGETS) * (ARM_COUNT + len(UNSEEN_TASKS))
print("Unlearning budgets:", UNLEARN_BUDGETS)
print("Image budgets:", BUDGETS, "| recovery steps each:", RECOVERY_STEPS)
print("Generator during recovery:", "frozen" if FREEZE_GENERATOR else "trainable")
print("Total recovery runs:", RUNS)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID`, and `R2_SECRET_ACCESS_KEY` through Colab Secrets
or environment variables.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)

## 4. Load the study and the source model

In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
path = store.download(PREFIX + "/study.json", CACHE / "cache")
plan = json.loads(path.read_text())
if plan["code_version"] != CODE_VERSION:
    raise ValueError("This study requires a different fixed code revision.")
from uncle.learning_diagnostic import file_hash
if file_hash(REPO / "uncle/task_partition.json") != plan["partition_sha256"]:
    raise ValueError("Class partition differs from the saved study.")

storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = tuple(plan["orders"][ORDER])
NEW_TASK = plan["new_tasks"][0]
CLASSES = plan["config"]["classes_per_task"]
EVAL_BATCH = plan["config"]["eval_batch_size"]

def fetch_tasks(include):
    """Build the task datasets, clearing a broken download if there is one."""
    import shutil, zipfile
    archive = Path(DATA_ROOT).parent / "tiny-imagenet-200.zip"
    for attempt in (1, 2):
        try:
            return build_tasks(root=DATA_ROOT,
                               partition_path=REPO / "uncle/task_partition.json",
                               partition_seed=42, include=include, download=True)
        except (zipfile.BadZipFile, EOFError, OSError) as problem:
            if attempt == 2:
                raise
            print("The Tiny ImageNet download is unusable:", problem)
            if archive.exists():
                head = archive.open("rb").read(4)
                print(f"   {archive} is {archive.stat().st_size} bytes, starts with {head!r}")
                archive.unlink()
            shutil.rmtree(Path(DATA_ROOT), ignore_errors=True)
            print("   Cleared it. Downloading again.")

tasks = fetch_tasks(ORDER_TASKS + tuple(UNSEEN_TASKS))

source_job = next(j for j in ALL_JOBS if j["id"] == f"sources/{ORDER}/seed_{SEED}")
branch_job = next(j for j in ALL_JOBS
                  if j["id"] == f"branches/{ORDER}/seed_{SEED}/forget_{FORGET_TASK}/learn_{NEW_TASK}")
state = storage.load(source_job)
if state is None or state["report"]["status"] != "complete":
    raise RuntimeError(f"Source model for seed {SEED} is not finished in R2.")

REQUESTS = tuple(("learn", t) for t in ORDER_TASKS)
REQUESTS += tuple(tuple(op) for op in branch_job["operations"])
PROTECT = [t for t in state["seen"] if t != FORGET_TASK]
print("Source:", source_job["id"], "| forgetting task", FORGET_TASK)

## 5. Measuring, and choosing the images

`inspect` reports four things about a set of generated weights.

- **accuracy** on the task's held-out test images.
- **input sensitivity**: how much the ten output scores move between different images. A
  model that has stopped reading its input has nothing for gradient descent to follow.
- **classes used**: how many of the ten classes the model actually predicts at least once
  across the whole test set. This is the collapse check. A model that answers one class
  for every image scores exactly 10% on a balanced set, which is also what guessing
  scores, so accuracy alone cannot separate the two. One class used means collapse.
- **top class share**: the fraction of all predictions that went to the single most
  common class. 0.1 means evenly spread, 1.0 means total collapse.

`sample_indices` picks an equal number of images from each class.

In [ ]:
from torch.func import functional_call
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F

def inspect(hypernet, target, trainer, code_task, data_task):
    """Score the weights produced by `code_task` on `data_task`'s images.

    The two differ in the fresh-start and stats-only arms, where a brand new
    code is trained on task 3's data, so its weights must still be judged on
    task 3's images.
    """
    # Evaluation mode matters more than usual here. In training mode
    # BatchNorm normalises using each batch's own statistics, which rescales a
    # nearly dead signal back up to unit size and hides exactly the collapse
    # this notebook is measuring. The previous mode is put back afterwards so
    # a measurement taken mid-training does not stop the training.
    was_training = hypernet.training
    hypernet.eval(); target.eval()
    weights = hypernet.weights_for(code_task)
    buffers = {n: v.clone() for n, v in trainer.task_buffers[code_task].items()}
    loader = DataLoader(tasks[data_task]["test"], batch_size=EVAL_BATCH)
    correct = total = 0
    lowest = highest = None
    # How often each class is predicted, over the whole test set. This is what
    # separates a collapsed model from a guessing one: both score 10%, but a
    # collapsed model puts every prediction in one or two bins.
    predicted = torch.zeros(CLASSES, dtype=torch.long)
    with torch.no_grad():
        for images, labels in loader:
            scores = functional_call(target, (weights, buffers),
                                     (images.to(trainer.device),), strict=True)
            guesses = scores.argmax(1)
            correct += (guesses == labels.to(trainer.device)).sum().item()
            total += labels.numel()
            predicted += torch.bincount(guesses.cpu(), minlength=CLASSES)
            low, high = scores.min(dim=0).values, scores.max(dim=0).values
            lowest = low if lowest is None else torch.minimum(lowest, low)
            highest = high if highest is None else torch.maximum(highest, high)
    if was_training:
        hypernet.train(); target.train()
    share = predicted.double() / max(int(predicted.sum()), 1)
    return {"accuracy": 100.0 * correct / total,
            "input_sensitivity": (highest - lowest).max().item(),
            "classes_used": int((predicted > 0).sum()),
            "top_class_share": float(share.max())}

def sample_indices(dataset, per_class, seed):
    """An equal number of images from each class, chosen reproducibly.

    ClassTask keeps its labels in `.targets`, so this costs nothing: no image
    is opened just to find out which class it belongs to.
    """
    generator = torch.Generator().manual_seed(seed)
    chosen = []
    for label in range(CLASSES):
        here = [i for i, value in enumerate(dataset.targets) if value == label]
        order = torch.randperm(len(here), generator=generator)[:per_class]
        chosen.extend(here[int(i)] for i in order)
    return chosen

## 6. The recovery step

New code, not a copy: the study has no recovery operation. It is an ordinary supervised
fine-tune of the generator and one task code, with the method's own protection term so the
other seven tasks are held still.

In [ ]:
def recover(trainer, code_task, data_task, protected, indices, steps, on_step):
    """Train the generator and one task code on a small sample of images.

    This is new code, not a copy of anything in the repository: the study has
    no recovery operation. It is an ordinary supervised fine-tune, with the
    same protection term the method uses elsewhere so the other tasks are
    held still while this one is pulled back.
    """
    hypernet, config = trainer.hypernet, trainer.config
    snapshot = hypernet.snapshot()
    trainer._reference = {}

    code = hypernet.task_codes[code_task]
    # With the generator frozen only the 32-number embedding can move, so
    # whatever accuracy appears was already reachable inside the generator.
    trainable = [code] if FREEZE_GENERATOR else [*hypernet.generator_parameters(), code]
    hypernet.requires_grad_(False)
    for parameter in trainable:
        parameter.requires_grad_(True)
    optimizer = torch.optim.Adam(trainable, lr=config.learning_rate)

    subset = Subset(tasks[data_task]["train"], indices)
    loader = DataLoader(subset, batch_size=min(config.batch_size, len(subset)),
                        shuffle=True, drop_last=False,
                        generator=torch.Generator().manual_seed(config.seed))

    hypernet.train(); trainer.target.train()
    on_step(0)
    step = 0
    while step < steps:
        for images, labels in loader:
            if step >= steps:
                break
            weights = hypernet.weights_for(code_task)
            scores = functional_call(trainer.target,
                                     (weights, trainer.task_buffers[code_task]),
                                     (images.to(trainer.device),), strict=True)
            loss = F.cross_entropy(scores, labels.to(trainer.device))
            # A frozen generator cannot drift, so the other tasks are safe
            # without paying for the protection term.
            if protected and not FREEZE_GENERATOR:
                loss = loss + config.beta * trainer.preserve(protected, snapshot)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            step += 1
            if step % MEASURE_EVERY == 0:
                with torch.no_grad():
                    on_step(step)
                hypernet.train(); trainer.target.train()
    trainer._reference = {}

## 7. Unlearn at each stopping point, then try every arm

For each entry in `UNLEARN_BUDGETS` the model is rebuilt from the saved source checkpoint
and unlearned from scratch. Every arm at that stopping point then starts from exactly that
state, so differences between arms come from the arm and nothing else, and differences
between stopping points come from the stopping point and nothing else.

In [ ]:
import copy, gc
from uncle.study_training import run_request

def build():
    config = Config(**{**plan["config"], "seed": SEED, "device": DEVICE,
                       "requests": REQUESTS, "burn_in": max(UNLEARN_BUDGETS)})
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=True)
    checkpoint.restore(state, hypernet=hypernet, uncle=trainer)
    restore_rng(state["all_rng"])
    return config, target, hypernet, trainer

# The four combinations of the two things unlearning leaves behind: task 3's
# code, and its BatchNorm statistics. With all four you can tell what each
# contributes on its own, and whether they only help together.
#   name            keeps the old code?   whose BatchNorm statistics?
ARMS = [("attacker",    True,  "task"),
        ("code only",   True,  "fresh"),
        ("stats only",  False, "task"),
        ("fresh start", False, "fresh")]

# before and after unlearning, per stopping point
BASELINE = {}
results = []

with Dashboard("Unlearning at each stopping point, then recovering") as board:
    work = board.bar(RUNS, "Recovery runs", unit="run")

    for unlearn_steps in UNLEARN_BUDGETS:
        board.note(f"unlearning task {FORGET_TASK} for {unlearn_steps} steps")
        print(f"\n{'=' * 72}\nUNLEARNING FOR {unlearn_steps} STEPS\n{'=' * 72}")

        # Free the previous stopping point's model before building the next
        # one. The generator has about 134 million parameters and a full copy
        # of it is held in UNLEARNED, so letting two settings overlap in
        # memory is what would run a smaller GPU out of room.
        for name in ("hypernet", "trainer", "target", "config",
                     "UNLEARNED", "BUFFERS", "FRESH_BUFFERS"):
            globals().pop(name, None)
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

        # Rebuilt from the saved checkpoint every time, so a 100-step run is
        # never a continuation of the 6-step one.
        config, target, hypernet, trainer = build()
        before = inspect(hypernet, target, trainer, FORGET_TASK, FORGET_TASK)
        run_request(trainer, "forget", FORGET_TASK, PROTECT, None, lambda row: None,
                    lambda active: None, forget_interval=unlearn_steps,
                    evaluation_interval=10 ** 9)
        after = inspect(hypernet, target, trainer, FORGET_TASK, FORGET_TASK)
        BASELINE[unlearn_steps] = {"before": before, "after": after}

        UNLEARNED = copy.deepcopy(hypernet.state_dict())
        BUFFERS = {t: {n: v.clone() for n, v in b.items()}
                   for t, b in trainer.task_buffers.items()}
        FRESH_BUFFERS = {n: v.clone() for n, v in trainer.buffer_template.items()}
        print(f"before unlearning : {before['accuracy']:5.1f}%  "
              f"sensitivity {before['input_sensitivity']:.3e}  "
              f"classes used {before['classes_used']}/{CLASSES}")
        print(f"after unlearning  : {after['accuracy']:5.1f}%  "
              f"sensitivity {after['input_sensitivity']:.3e}  "
              f"classes used {after['classes_used']}/{CLASSES}")

        def reset_to_unlearned():
            """Put the unlearned model back, so every arm starts identically.

            A previous arm may have added a task code that the saved state does
            not contain, and load_state_dict refuses an unexpected one, so drop
            those first.
            """
            for name in list(hypernet.task_codes):
                if f"task_codes.{name}" not in UNLEARNED:
                    del hypernet.task_codes[name]
            hypernet.load_state_dict(UNLEARNED)
            trainer.task_buffers = {t: {n: v.clone() for n, v in b.items()}
                                    for t, b in BUFFERS.items()}

        for budget in BUDGETS:
            indices = sample_indices(tasks[FORGET_TASK]["train"], budget, seed=1000 + budget)
            for arm, keep_code, stats in ARMS:
                board.note(f"{unlearn_steps} steps | {arm}, {budget} image(s) per class")
                print(f"\n=== {arm}: {budget} image(s) per class, "
                      f"{len(indices)} images total, unlearned {unlearn_steps} steps ===")
                reset_to_unlearned()

                if keep_code:
                    code_task = FORGET_TASK
                else:
                    # A brand new code: no history with task 3 at all.
                    code_task = (f"{FORGET_TASK}_fresh_{arm.replace(' ', '_')}"
                                 f"_{budget}_{unlearn_steps}")
                    if code_task in hypernet.task_codes:
                        del hypernet.task_codes[code_task]
                    hypernet.add_task(code_task)
                # "task" hands over the statistics unlearning left behind, which
                # were computed from task 3's own images. "fresh" is a blank slate.
                source = BUFFERS[FORGET_TASK] if stats == "task" else FRESH_BUFFERS
                trainer.task_buffers[code_task] = {n: v.clone() for n, v in source.items()}

                rows = []
                def note(step, code_task=code_task):
                    row = {"step": step,
                           **inspect(hypernet, target, trainer, code_task, FORGET_TASK)}
                    rows.append(row)
                    print(f"    step {step:>3} | accuracy {row['accuracy']:5.1f}% | "
                          f"classes {row['classes_used']:>2}/{CLASSES} | "
                          f"sensitivity {row['input_sensitivity']:.3e}")

                recover(trainer, code_task, FORGET_TASK,
                        PROTECT if PROTECT_DURING_RECOVERY else [],
                        indices, RECOVERY_STEPS, note)
                others = {t: inspect(hypernet, target, trainer, t, t)["accuracy"]
                          for t in PROTECT}
                results.append({"unlearn_steps": unlearn_steps, "arm": arm,
                                "budget": budget, "images": len(indices),
                                "rows": rows, "other_tasks": others,
                                "others_mean": sum(others.values()) / len(others)})
                work.update(1)
                gc.collect()
                if DEVICE == "cuda":
                    torch.cuda.empty_cache()

        # The control: how fast does this same model learn a task it has never
        # seen, given the same budget? Nothing about task 3 is involved.
        for budget in BUDGETS:
            for unseen in UNSEEN_TASKS:
                board.note(f"{unlearn_steps} steps | unseen task {unseen}, "
                           f"{budget} image(s) per class")
                print(f"\n=== unseen task {unseen}: {budget} image(s) per class, "
                      f"unlearned {unlearn_steps} steps ===")
                reset_to_unlearned()

                code_task = f"unseen_{unseen}_{budget}_{unlearn_steps}"
                hypernet.add_task(code_task)
                trainer.task_buffers[code_task] = {n: v.clone()
                                                   for n, v in FRESH_BUFFERS.items()}
                indices = sample_indices(tasks[unseen]["train"], budget, seed=2000 + budget)

                rows = []
                def note(step, code_task=code_task, unseen=unseen):
                    row = {"step": step,
                           **inspect(hypernet, target, trainer, code_task, unseen)}
                    rows.append(row)
                    print(f"    step {step:>3} | accuracy {row['accuracy']:5.1f}% | "
                          f"classes {row['classes_used']:>2}/{CLASSES} | "
                          f"sensitivity {row['input_sensitivity']:.3e}")

                recover(trainer, code_task, unseen,
                        PROTECT if PROTECT_DURING_RECOVERY else [],
                        indices, RECOVERY_STEPS, note)
                results.append({"unlearn_steps": unlearn_steps, "arm": f"unseen {unseen}",
                                "budget": budget, "images": len(indices), "rows": rows,
                                "other_tasks": {}, "others_mean": float("nan")})
                work.update(1)
                gc.collect()
                if DEVICE == "cuda":
                    torch.cuda.empty_cache()

    work.close()

print(f"\nFinished {len(results)} recovery runs "
      f"across {len(UNLEARN_BUDGETS)} unlearning budgets.")

## 8. Compare

In [ ]:
by_key = {(r["unlearn_steps"], r["arm"], r["budget"]): r for r in results}
chance = 100.0 / CLASSES
ARM_NAMES = ("attacker", "code only", "stats only", "fresh start")

def best(run):
    """The highest accuracy reached, not the last one.

    Over a long run on few images the model overfits and the final number
    drifts below the peak. For "could someone restore this task", the peak is
    the honest figure: it is what an attacker would stop at.
    """
    return max(r["accuracy"] for r in run["rows"])

def wake_step(run, floor=0.1):
    """First step where the model starts responding to its input again.

    Until the output changes between different images there is no learning
    signal to follow, so these steps are dead time rather than slow progress.
    """
    for row in run["rows"]:
        if row["input_sensitivity"] > floor:
            return row["step"]
    return None

def steps_to(run, target):
    """First step whose accuracy reaches `target`, or None if it never does."""
    for row in run["rows"]:
        if row["accuracy"] >= target:
            return row["step"]
    return None

def mean_lift(run):
    """Average accuracy above chance across the whole run.

    The curves wobble by several points, so any single measurement is a poor
    summary. This uses all of them.
    """
    return sum(max(row["accuracy"] - chance, 0.0) for row in run["rows"]) / len(run["rows"])

def collapsed(run):
    """Did the model use only one class at any measured point?"""
    return min(row["classes_used"] for row in run["rows"])

print("Protocol:", "generator FROZEN, embedding only (the authors' own setting)"
      if FREEZE_GENERATOR else "generator trainable (what an attacker could do)")
if FREEZE_GENERATOR:
    print("  Note: with the generator frozen, a fresh code sits at chance by")
    print("  construction. Read 'fresh start' as a floor, not as a control.")

for unlearn_steps in UNLEARN_BUDGETS:
    base = BASELINE[unlearn_steps]
    print(f"\n{'=' * 78}")
    print(f"UNLEARNED FOR {unlearn_steps} STEPS")
    print(f"{'=' * 78}")
    print(f"task {FORGET_TASK} scored {base['before']['accuracy']:.1f}% before unlearning "
          f"and {base['after']['accuracy']:.1f}% straight after. Chance is {chance:.0f}%.")
    print(f"straight after unlearning the model used "
          f"{base['after']['classes_used']}/{CLASSES} classes, with "
          f"{100 * base['after']['top_class_share']:.0f}% of predictions in the commonest one.")

    print("\nBEST ACCURACY REACHED")
    header = f"{'budget':>7} {'images':>7}  " + " ".join(f"{a:>12}" for a in ARM_NAMES)
    print(header + f"   {'others kept':>11}")
    print("-" * len(header + "   others kept"))
    for budget in BUDGETS:
        row = [by_key[(unlearn_steps, arm, budget)] for arm in ARM_NAMES]
        peaks = [best(r) for r in row]
        print(f"{budget:>7} {row[0]['images']:>7}  "
              + " ".join(f"{p:11.1f}%" for p in peaks)
              + f"   {row[0]['others_mean']:10.1f}%")

    print("\nFEWEST CLASSES THE MODEL EVER USED (1 means it collapsed onto one answer)")
    print(f"{'budget':>7}  " + " ".join(f"{a:>12}" for a in ARM_NAMES))
    print("-" * (9 + 13 * len(ARM_NAMES)))
    for budget in BUDGETS:
        row = [by_key[(unlearn_steps, arm, budget)] for arm in ARM_NAMES]
        print(f"{budget:>7}  " + " ".join(f"{collapsed(r):>12}" for r in row))
    print("  A run that stays at 1 class scored its accuracy by always answering the")
    print("  same thing. That is collapse, and it is not the same as forgetting.")

    print("\nVERDICT")
    gaps, bn_gaps, any_learning = [], [], False
    for budget in BUDGETS:
        a = best(by_key[(unlearn_steps, "attacker", budget)])
        f = best(by_key[(unlearn_steps, "fresh start", budget)])
        c = best(by_key[(unlearn_steps, "code only", budget)])
        gaps.append(a - f)
        bn_gaps.append(a - c)
        any_learning = any_learning or max(a, f, c) > chance + 2.0

    if not any_learning:
        print("  Nothing recovered in any arm. The learning signal has vanished.")
        print("  This does NOT show the information was removed, only that gradient")
        print("  descent could not reach it. Say exactly that, and no more.")
    else:
        mean_gap = sum(gaps) / len(gaps)
        print(f"  Attacker beats a fresh start by {mean_gap:+.1f} points on average.")
        if mean_gap > 3.0:
            print("  Something about task 3 survived unlearning. The model recovers it")
            print("  faster than it learns it from nothing, using the same images and")
            print("  the same number of steps. A privacy claim for this method fails.")
        elif mean_gap < -3.0:
            print("  A fresh code does BETTER than the old one, so unlearning left the")
            print("  old code actively harder to train than an unused one.")
        else:
            print("  The two are close, so no usable trace of task 3 is left in the code.")
        mean_bn = sum(bn_gaps) / len(bn_gaps)
        print(f"\n  Keeping the BatchNorm statistics is worth {mean_bn:+.1f} points on average.")
        if mean_bn > 3.0:
            print("  Those statistics are derived from task 3's images and unlearning")
            print("  never touches them. That is a data remnant the method does not")
            print("  attempt to remove, and it is worth reporting on its own.")

    print("\nCHEAP COMPARED TO WHAT? tasks this model has never seen")
    print(f"{'budget':>7}  {'task 3, fresh code':>19}  " +
          "  ".join(f"{'task ' + u:>8}" for u in UNSEEN_TASKS) + f"  {'their mean':>11}")
    print("-" * (32 + 10 * len(UNSEEN_TASKS) + 13))
    unseen_means = {}
    for budget in BUDGETS:
        peaks = [best(by_key[(unlearn_steps, f"unseen {u}", budget)]) for u in UNSEEN_TASKS]
        unseen_means[budget] = sum(peaks) / len(peaks)
        own = best(by_key[(unlearn_steps, "fresh start", budget)])
        print(f"{budget:>7}  {own:18.1f}%  " + "  ".join(f"{p:7.1f}%" for p in peaks)
              + f"  {unseen_means[budget]:10.1f}%")

    gap_unseen = sum(best(by_key[(unlearn_steps, "fresh start", b)]) - unseen_means[b]
                     for b in BUDGETS) / len(BUDGETS)
    print(f"\n  Task 3 with a fresh code beats an unseen task by {gap_unseen:+.1f} points on average.")
    if gap_unseen > 3.0:
        print("  Task 3 is easier for this model than a task it never saw, even from a")
        print("  brand new code. Knowledge of task 3 is still in the shared generator,")
        print("  which unlearning was supposed to remove. This is the privacy result.")
    elif gap_unseen < -3.0:
        print("  An unseen task is EASIER than task 3, so task 3 is simply a hard task")
        print("  and no residual advantage is visible.")
    else:
        print("  Task 3 is no easier than a task the model never saw. So the cheap")
        print("  recovery reflects how easily this model learns anything, not leftover")
        print("  knowledge of task 3. Say that plainly rather than claiming a leak.")
    print("  Treat anything within 3 points as a tie: identical runs differ by about that much.")

    print("\nWHAT EACH LEFTOVER IS WORTH (best accuracy reached)")
    print(f"{'budget':>7}  {'old code + old stats':>21} {'old code only':>14} "
          f"{'old stats only':>15} {'neither':>9}")
    print("-" * 72)
    for budget in BUDGETS:
        cells = [best(by_key[(unlearn_steps, a, budget)]) for a in
                 ("attacker", "code only", "stats only", "fresh start")]
        print(f"{budget:>7}  {cells[0]:20.1f}% {cells[1]:13.1f}% "
              f"{cells[2]:14.1f}% {cells[3]:8.1f}%")
    print("\n  Reading across: how much each leftover is worth on its own, against")
    print("  keeping neither. A difference under 3 points is a tie.")

    halfway = chance + 0.5 * (base["before"]["accuracy"] - chance)
    print(f"\nHOW FAST, AND WHY (halfway back is {halfway:.1f}%)")
    print(f"{'budget':>7} {'arm':<14} {'wakes at':>9} {'reaches halfway':>16} "
          f"{'after waking':>13} {'mean lift':>10}")
    print("-" * 76)
    speed_arms = list(ARM_NAMES) + [f"unseen {u}" for u in UNSEEN_TASKS]
    for budget in BUDGETS:
        for arm in speed_arms:
            run = by_key.get((unlearn_steps, arm, budget))
            if run is None:
                continue
            wake, hit = wake_step(run), steps_to(run, halfway)
            # Named so it cannot shadow the `after` measurement above. The
            # earlier version of this notebook reused `after` here, which
            # overwrote the after-unlearning reading before it was saved.
            gap_after_wake = "-" if (wake is None or hit is None) else str(max(hit - wake, 0))
            print(f"{budget:>7} {arm:<14} {str(wake) if wake is not None else 'never':>9}"
                  f" {str(hit) if hit is not None else 'never':>16} {gap_after_wake:>13}"
                  f" {mean_lift(run):9.1f}")
    print("\n  wakes at         the step where the model starts reacting to its input")
    print("  reaches halfway  the step where accuracy first gets halfway back")
    print("  after waking     the two subtracted: learning speed once it can learn")
    print("  mean lift        average accuracy above chance, over the whole run")
    print("\n  If the arms differ in 'wakes at' but match in 'after waking', then the")
    print("  cost of unlearning is dead time, not damage to how well it relearns.")

# -- does stopping earlier leave more behind? --------------------------------
if len(UNLEARN_BUDGETS) > 1:
    print(f"\n{'=' * 78}")
    print("DOES STOPPING EARLIER LEAVE MORE BEHIND?")
    print(f"{'=' * 78}")
    print("The task is already at chance at every stopping point here. So any")
    print("difference below is about how reachable it still is, not about how")
    print("much was removed.")
    print(f"\n{'arm':<14} {'budget':>7}  " +
          "  ".join(f"{'stop at ' + str(s):>12}" for s in UNLEARN_BUDGETS))
    print("-" * (23 + 14 * len(UNLEARN_BUDGETS)))
    for arm in ARM_NAMES:
        for budget in BUDGETS:
            peaks = [best(by_key[(s, arm, budget)]) for s in UNLEARN_BUDGETS]
            print(f"{arm:<14} {budget:>7}  " + "  ".join(f"{p:11.1f}%" for p in peaks))

    early, late = UNLEARN_BUDGETS[0], UNLEARN_BUDGETS[-1]
    diffs = [best(by_key[(early, "attacker", b)]) - best(by_key[(late, "attacker", b)])
             for b in BUDGETS]
    mean_diff = sum(diffs) / len(diffs)
    print(f"\n  Attacker arm: stopping at {early} recovers {mean_diff:+.1f} points more")
    print(f"  than stopping at {late}, averaged over the image budgets.")
    if mean_diff > 3.0:
        print(f"  The extra steps past {early} do not remove more, but they do make what")
        print("  is left harder to reach. That is irreversibility, not forgetting, and")
        print("  it is the claim the step-6 accuracy trace points at.")
    elif mean_diff < -3.0:
        print(f"  Stopping at {late} leaves MORE recoverable, which is the opposite of")
        print("  what the direction measurement predicts. Check this before reporting it.")
    else:
        print("  No difference worth reporting. The extra steps change neither how much")
        print("  was removed nor how reachable it is.")

## 9. Chart

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime, timezone

EXPORT = CACHE / "diagnostics" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EXPORT.mkdir(parents=True, exist_ok=True)

colours = {"attacker": "#b8322a", "code only": "#4c956c",
           "stats only": "#d08c60", "fresh start": "#245d99"}

rows_n, cols_n = len(UNLEARN_BUDGETS), len(BUDGETS)
figure, axes = plt.subplots(rows_n, cols_n, figsize=(5 * cols_n, 4.4 * rows_n),
                            sharey=True, squeeze=False)

for row_index, unlearn_steps in enumerate(UNLEARN_BUDGETS):
    base = BASELINE[unlearn_steps]
    for col_index, budget in enumerate(BUDGETS):
        axis = axes[row_index][col_index]
        for arm in colours:
            run = by_key[(unlearn_steps, arm, budget)]
            axis.plot([r["step"] for r in run["rows"]],
                      [r["accuracy"] for r in run["rows"]],
                      color=colours[arm], marker="o", linewidth=1.8, label=arm)
        axis.axhline(base["before"]["accuracy"], color="black", linestyle="--",
                     linewidth=1, label="before unlearning")
        axis.axhline(chance, color="grey", linestyle=":", linewidth=1, label="chance")
        axis.set_title(f"unlearned {unlearn_steps} steps | {budget} image(s) per class")
        axis.set_xlabel("recovery step")
    axes[row_index][0].set_ylabel("task " + FORGET_TASK + " accuracy (%)")
axes[0][0].legend(fontsize=8)

figure.suptitle(f"{STUDY_ID} | seed {SEED} | recovering task {FORGET_TASK}")
figure.tight_layout()
figure.savefig(EXPORT / "recovery.png", dpi=180)
figure.savefig(EXPORT / "recovery.svg")
plt.show()
print("Figure written to", EXPORT)

## 10. Save

In [ ]:
import csv

summary = {
    "study_id": STUDY_ID, "code_version": CODE_VERSION, "order": ORDER, "seed": SEED,
    "forget_task": FORGET_TASK, "unlearn_budgets": UNLEARN_BUDGETS,
    "budgets": BUDGETS, "recovery_steps": RECOVERY_STEPS,
    "measure_every": MEASURE_EVERY,
    "unseen_tasks": UNSEEN_TASKS,
    "protected_during_recovery": PROTECT_DURING_RECOVERY,
    "freeze_generator": FREEZE_GENERATOR,
    # Keyed by stopping point. The previous version of this notebook stored a
    # single pair here and a later loop overwrote the after-unlearning value
    # with a formatting string before it was written out.
    "baseline": {str(s): BASELINE[s] for s in UNLEARN_BUDGETS},
    "classes_per_task": CLASSES, "beta": plan["config"]["beta"],
    "results": results,
    "reading": ("Every arm starts from the same unlearned model, at each stopping point "
                "in unlearn_budgets, and gets the same images and the same number of "
                "steps. The attacker arm keeps task 3's code and BatchNorm statistics; "
                "code only keeps the code; stats only keeps the statistics; fresh start "
                "keeps neither. Recovery images come from task 3's own training set. "
                "classes_used counts how many distinct classes the model predicted "
                "across the test set: 1 means it collapsed onto a single answer, which "
                "scores chance on a balanced set without forgetting anything."),
}
(EXPORT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

with (EXPORT / "recovery.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=["unlearn_steps", "arm", "budget", "images",
                                                "step", "accuracy", "input_sensitivity",
                                                "classes_used", "top_class_share"])
    writer.writeheader()
    for run in results:
        for row in run["rows"]:
            writer.writerow({"unlearn_steps": run["unlearn_steps"], "arm": run["arm"],
                             "budget": run["budget"], "images": run["images"], **row})

print("Saved locally:")
for item in sorted(EXPORT.iterdir()):
    print("  ", item)

if SAVE_TO_R2:
    destination = PREFIX + "/diagnostics/" + EXPORT.name
    with Dashboard("Saving results") as board:
        files = sorted(EXPORT.iterdir())
        sending = board.bar(len(files), "Upload diagnostics")
        for item in files:
            sending.set_postfix_str(item.name)
            store.upload(item, destination + "/" + item.name)
            sending.update(1)
        sending.close()
    print("Uploaded to", destination)
    SAVED_LOCATION = destination
else:
    SAVED_LOCATION = str(EXPORT)
    print("\nSAVE_TO_R2 is off, so the only copy is inside this runtime.")

EXPECTED_FILES = ["recovery.csv", "recovery.png", "recovery.svg", "summary.json"]
SAVE_VERIFIED = sorted(item.name for item in EXPORT.iterdir()) == EXPECTED_FILES
print("\nAll expected files present:", SAVE_VERIFIED)

## 11. Hand the GPU back

In [ ]:
if not SAVE_VERIFIED:
    print("Not releasing the GPU: some output files are missing.")
    print("Rerun the save section before closing this runtime.")
elif not RELEASE_GPU_WHEN_DONE or not IN_COLAB:
    print("Results saved to:", SAVED_LOCATION)
else:
    print("Results saved to:", SAVED_LOCATION)
    print("Share that path.")
    import gc
    for name in ("results", "state", "tasks", "by_key", "BASELINE"):
        globals().pop(name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    from google.colab import runtime
    print("Releasing the Colab runtime now.")
    runtime.unassign()

## What to tell your mentor

Send me the two VERDICT blocks from section 8, one per stopping point, and the
"DOES STOPPING EARLIER LEAVE MORE BEHIND?" table at the end.

**If the attacker arm wins**, you have a privacy result: the method's own success metric
says the task is forgotten, while a handful of its original images bring it back faster
than learning it from nothing would. The metrics in the paper cannot see this.

**If the BatchNorm gap is large**, you have a second, independent result: unlearning never
touches the running statistics, which are derived from the forgotten task's images, so a
data remnant is left behind by construction. This one does not even need the recovery to
succeed to be worth reporting.

**If recovery works at 6 steps but not at 100**, you have the third result: the 94 extra
steps do not remove more, they only put what is left out of reach. That is the recoverable
version of the claim the step-6 accuracy trace already points at.

**If nothing recovers at either stopping point**, say only that gradient descent could not
reach the task from a model that has stopped responding to its input. That is not the same
as the information being gone, and claiming otherwise would be the easiest thing for a
reviewer to attack. Check the "fewest classes used" table before writing anything: if the
arms sat at one class throughout, the run measured collapse, not removal.

**What this does not cover.** One seed, one task, one order, recovery from the task's own
training images rather than unseen ones. Enough for a finding, not enough for a general
claim about the method.